# Entrainement Et Evaluation LLM Support

Ce notebook charge les jeux de donnees prepares, evalue un LLM open-weight de base, le personnalise, puis compare les deux versions sur le jeu de test.


## Installer Les Dependances

Installez les dependances du projet dans l'environnement actif du notebook avant d'executer le reste du notebook.


In [ ]:
%pip install -r requirements.txt


## Imports

Ajoute les bibliotheques necessaires pour charger les jeux de donnees, executer le modele et calculer les metriques d'evaluation.


In [3]:
import json
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType
from support_classification.evaluation import evaluate_model, predict_queue
from support_classification.prompts import build_prompt
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from support_classification.paths import LABELS_PATH, PROJECT_ROOT, TRAIN_DATASET_PATH, TEST_DATASET_PATH

# Affiche les textes de prompt longs sans troncature lors de l'inspection des jeux de donnees.
pd.set_option('display.max_colwidth', 120)


ModuleNotFoundError: No module named 'torch'

## Charger Les Jeux De Donnees Prepares

Charge les fichiers CSV train et test produits par le notebook de preparation.


In [3]:
train_dataset = pd.read_csv(TRAIN_DATASET_PATH)
test_dataset = pd.read_csv(TEST_DATASET_PATH)
valid_labels = json.loads(LABELS_PATH.read_text(encoding='utf-8'))['labels']

print('Train shape:', train_dataset.shape)
print('Test shape:', test_dataset.shape)
print('Valid labels:', valid_labels)


Train shape: (479, 2)
Test shape: (120, 2)


## Charger Le Modele De Base

Charge le modele open-weight choisi et le prepare pour l'etape de personnalisation.


In [ ]:
# Choix initial de modele conserve ici comme option de repli.
# Identifiant officiel correct : mistralai/Ministral-3-8B-Instruct-2512
# Ce modele instruct open-weight est multilingue et suffisamment solide pour la classification de tickets de support,
# il reste donc une bonne solution de secours si le plus petit modele n'atteint pas le score cible plus tard.
# base_model_id = 'mistralai/Ministral-3-8B-Instruct-2512'

# Commence avec un modele instruct plus petit car le jeu d'entrainement est modeste (479 exemples).
# Qwen2.5-1.5B-Instruct est bien plus leger a fine-tuner, moins couteux a executer, et reste adapte a la classification de texte multilingue.
base_model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Definit un token de padding pour le batching pendant l'entrainement.
# Reutilise le token EOS car de nombreux tokenizers de LLM causaux ne fournissent pas de pad token par defaut.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)

# Garde la configuration du modele alignee avec le comportement de padding du tokenizer.
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# Garde les identifiants des tokens speciaux alignes meme lorsque certains ne sont pas definis sur le tokenizer.
# Cela evite des avertissements ulterieurs lorsque les utilitaires d'entrainement resynchronisent les configurations du modele et du tokenizer.
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

# Desactive le cache KV pendant l'entrainement pour eviter les incompatibilites avec le gradient checkpointing ou les utilitaires de fine-tuning.
model.config.use_cache = False

print('Loaded model:', base_model_id)


## Definir Le Prompt D'Inference

Reutilise la meme structure d'entree que dans le notebook de preparation afin que le modele voie des prompts coherents.


In [ ]:
# Les helpers de prompt sont importes depuis support_classification.prompts afin que le notebook d'entrainement utilise exactement
# le meme template que le notebook de preparation des donnees.
# Les jeux de donnees traites contiennent deja ce format de prompt, et cet exemple montre comment
# le reconstruire plus tard pour de nouveaux tickets au moment de l'inference.
example_prompt = build_prompt(
    subject='Compatibility Inquiry',
    body='I recently purchased a MacBook Air M1 and need to know if it is compatible with my external wireless monitor setup.',
    language='en',
    business_type='Tech Online Store',
    valid_labels=valid_labels,
)

print(example_prompt)


You are a support ticket classification assistant.
Predict the ticket category from the information below.

Subject: Compatibility Inquiry
Body: I recently purchased a MacBook Air M1 and need to know if it is compatible with my external wireless monitor setup.
Language: en
Business type: Tech Online Store

Answer:


## Preparer Les Donnees De Fine-Tuning

Met en forme le jeu d'entrainement pour qu'il puisse etre utilise par le pipeline de personnalisation a l'etape suivante.


In [ ]:
# Ne conserve que le split d'entrainement pour l'etape de personnalisation.
# Le jeu de test reste intact et ne sera utilise que plus tard pendant l'evaluation.
train_sft_dataset = train_dataset.copy()

# Construit un champ texte unique pour le fine-tuning supervise tout en conservant les colonnes d'origine inchangees.
# Le prompt se termine deja par 'Answer:', nous ajoutons donc un espace avant le label attendu.
train_sft_dataset['text'] = train_sft_dataset.apply(
    lambda row: f"{row['prompt']} {row['answer']}",
    axis=1,
)

# Convertit le DataFrame prepare en Dataset Hugging Face, qui est le format attendu par SFTTrainer.
hf_train_sft_dataset = Dataset.from_pandas(
    train_sft_dataset[['text']],
    preserve_index=False,
)

display(train_sft_dataset[['prompt', 'answer', 'text']].head(2))
print(hf_train_sft_dataset)


## Fine-Tuner Avec LoRA

Configure une adaptation LoRA legere sur le split d'entrainement et sauvegarde les artefacts du modele personnalise.


In [ ]:
# Une petite configuration LoRA suffit pour ce projet et maintient un cout d'entrainement raisonnable.
# Les modules cibles choisis correspondent aux principales couches de projection utilisees par les modeles causaux Qwen2.5.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

# Sauvegarde l'adapter personnalise dans le projet pour pouvoir le reutiliser plus tard sans reentrainement.
adapter_output_dir = PROJECT_ROOT / 'artifacts' / 'qwen25-support-lora'
adapter_output_dir.mkdir(parents=True, exist_ok=True)

# Privilegie bf16 sur les GPU compatibles, sinon bascule sur fp16 avec CUDA et fp32 sur CPU.
bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled

# Active le gradient checkpointing pour reduire l'utilisation memoire pendant le fine-tuning.
model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

# Les hyperparametres d'entrainement sont volontairement simples et prudents pour un petit jeu supervise.
sft_config = SFTConfig(
    output_dir=str(adapter_output_dir),
    dataset_text_field='text',
    max_length=768,
    packing=False,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    bf16=bf16_enabled,
    fp16=fp16_enabled,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=hf_train_sft_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()
trainer.save_model(str(adapter_output_dir))

# Sauvegarde le tokenizer avec l'adapter LoRA afin que le modele fine-tune puisse etre recharge plus tard
# avec les memes parametres de tokenization.
tokenizer.save_pretrained(str(adapter_output_dir))

print('Saved personalized model artifacts to:', adapter_output_dir)


## Verifier Les Artefacts Sauvegardes

Verifie que les fichiers de l'adapter LoRA et du tokenizer ont bien ete sauvegardes apres l'entrainement.


In [ ]:
# Liste les fichiers sauvegardes pour confirmer que les artefacts de l'adapter et du tokenizer sont disponibles.
saved_artifacts = sorted(path.name for path in adapter_output_dir.iterdir())
print('Saved artifacts:')
for artifact_name in saved_artifacts:
    print('-', artifact_name)


## Lancer Une Verification Qualitative

Recharge l'adapter personnalise et inspecte quelques predictions comme verification rapide afin de confirmer que le modele sauvegarde se recharge correctement et produit des labels propres avant l'evaluation quantitative complete sur le jeu de test.


In [ ]:
# Recharge le tokenizer et l'adapter entraine depuis le disque pour verifier que les artefacts sauvegardes sont reutilisables.
inference_tokenizer = AutoTokenizer.from_pretrained(str(adapter_output_dir))
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)
personalized_model = PeftModel.from_pretrained(base_model_for_inference, str(adapter_output_dir))
personalized_model.eval()

# Inspecte quelques exemples d'entrainement pour confirmer que le modele personnalise produit un label propre.
qualitative_examples = train_dataset.head(3).copy()
qualitative_examples['prediction'] = qualitative_examples['prompt'].apply(
    lambda prompt: predict_queue(prompt, personalized_model, inference_tokenizer, valid_labels)
)
display(qualitative_examples[['prompt', 'answer', 'prediction']])


## Evaluer Les Deux Modeles

Execute la meme procedure de test sur le split de test partage et calcule le weighted F1-score pour les deux versions.


In [ ]:
# Recharge une copie propre du modele de base afin de le comparer de maniere equitable a l'adapter personnalise.
base_model_for_evaluation = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)
base_model_for_evaluation.eval()

base_test_predictions, base_weighted_f1 = evaluate_model(
    test_dataset,
    base_model_for_evaluation,
    tokenizer,
    valid_labels,
    'Base model',
)

personalized_test_predictions, personalized_weighted_f1 = evaluate_model(
    test_dataset,
    personalized_model,
    inference_tokenizer,
    valid_labels,
    'Personalized model',
)

display(personalized_test_predictions[['prompt', 'answer', 'prediction']].head(5))


## Comparer Les Resultats

Resume les scores du modele de base et du modele personnalise dans un tableau de comparaison compact.


In [ ]:
target_f1 = 0.92

comparison_df = pd.DataFrame(
    [
        {
            'model': 'Base model',
            'weighted_f1': base_weighted_f1,
            'target_92_reached': base_weighted_f1 >= target_f1,
        },
        {
            'model': 'Personalized model',
            'weighted_f1': personalized_weighted_f1,
            'target_92_reached': personalized_weighted_f1 >= target_f1,
        },
    ]
)
comparison_df['weighted_f1'] = comparison_df['weighted_f1'].map(lambda score: f'{score:.2%}')

f1_improvement = personalized_weighted_f1 - base_weighted_f1
print(f'Weighted F1 improvement: {f1_improvement:+.2%}')
print(f'Personalized model reached the 92% target: {personalized_weighted_f1 >= target_f1}')
display(comparison_df)
